# 03. Closed-loop 평가 (Isaac Lab + LeIsaac)

모듈 5는 fine-tune한 GR00T 모델을 [Isaac Lab](https://isaac-sim.github.io/IsaacLab/) + [LeIsaac](https://github.com/LightwheelAI/leisaac)로 시뮬레이터 안에서 평가합니다. SO-101 로봇 + 주방 씬에서 "pick up the orange" 명령 수행을 측정합니다.

이 노트북은 `run-isaaclab.sh`를 감싸는 얇은 안내 래퍼입니다 — 실제 eval은 노트북 셀이 아니라 스크립트가 띄우는 **컨테이너 안에서 수동으로 실행**합니다.

## 선행 조건

- fine-tuned 모델을 S3에서 `aws s3 sync`로 받아 두었음 (SageMaker Pipeline의 학습 잡이 압축 해제 상태로 직접 업로드한 결과물)
- GR00T Policy Server가 해당 모델로 떠 있음 (ZMQ, 기본 포트 `5555`)

## 1단계: 환경 준비 + 컨테이너 기동

`run-isaaclab.sh`가 하는 일:

1. `~/isaaclab-pkgs`에 `leisaac[gr00t]` + `lerobot`을 영속 설치 (한 번만, 이후 스킵)
2. SO-101 로봇 + 주방 씬 USD 에셋을 `~/leisaac-assets/`로 다운로드 (한 번만, 이후 스킵)
3. leisaac repo를 clone하고 GR00T N1.6 언어 키 / headless 키보드 이슈에 대한 패치를 적용
4. Isaac Lab 컨테이너를 기동 (인터랙티브 셸)

**인터랙티브 셸로 진입하므로 노트북 셀이 아니라 code-server 터미널에서 실행하세요.** 첫 실행은 패키지 설치 + 에셋 다운로드로 수 분이 걸릴 수 있습니다. 이후 실행은 마커 파일 덕분에 바로 컨테이너 기동으로 넘어갑니다.

In [ ]:
# run-isaaclab.sh는 인터랙티브 컨테이너 셸로 진입하므로 노트북 셀에서 실행하면 블로킹됩니다.
# code-server 터미널에서 직접 실행하세요:
#
#   cd ~/aws-physical-ai-recipes/e2e-workshop/groot/inference && bash run-isaaclab.sh

## 2단계: 컨테이너 **안에서** eval 실행

아래 명령은 노트북이 아니라 위에서 기동한 **컨테이너 셸**에서 실행합니다. `--policy_host`/`--policy_port`에 GR00T Policy Server 주소를 지정하세요 (기본값은 같은 호스트의 `localhost:5555`).

```shell
python scripts/evaluation/policy_inference.py \
    --task=LeIsaac-SO101-PickOrange-v0 \
    --eval_rounds=10 \
    --policy_type=gr00tn1.6 \
    --policy_host=localhost --policy_port=5555 \
    --policy_timeout_ms=5000 --policy_action_horizon=16 \
    --policy_language_instruction="Pick up the orange and place it on the plate" \
    --device=cuda --enable_cameras
```

`run-isaaclab.sh`를 조정할 수 있는 환경변수:

| 환경변수 | 기본값 | 의미 |
|----------|--------|------|
| `ISAAC_LAB_IMAGE` | `nvcr.io/nvidia/isaac-lab:2.3.0` | 사용할 Isaac Lab 컨테이너 이미지 |
| `LEISAAC_COMMIT` | (스크립트에 명시된 고정 커밋) | leisaac 리포를 고정할 커밋 |

## 결과 해석

- 컨테이너 콘솔 로그에 라운드별 성공/실패가 출력됩니다. `--eval_rounds`만큼 반복한 성공률로 정책 품질을 판단하세요.
- 실패가 많다면 GR00T Policy Server가 올바른 fine-tuned 모델(체크포인트)로 떠 있는지, `embodimentTag`가 태스크(SO-101 pick-orange)와 맞는지 먼저 확인하세요.
- 이 스테이지는 비디오를 저장하지 않습니다 (`policy_inference.py`가 recorder를 비활성화). 시각적으로 확인하려면 컨테이너의 인터랙티브 뷰포트로 직접 관찰하세요.
- 전체 closed-loop 평가 절차와 배경은 워크숍 가이드 **모듈 5**를 참고하세요.